# Networking, Ingress & Network Policies

## What's covered

- **The Kubernetes networking model** — the four hard requirements every cluster has to satisfy, and why "pods get real IPs" is a big deal
- **Pod IPs and pod CIDRs** — where the addresses come from, how each node gets a sub-range
- **What a CNI plugin actually does** — the second-by-second story of a pod getting on the network
- **The three communication paths** — pod-to-pod, pod-to-Service, pod-to-external — and how the kernel routes each
- **Ingress** — why a per-Service LoadBalancer doesn't scale, and what an Ingress resource gives you that Services can't
- **Ingress controllers** — `ingress-nginx`, Traefik, HAProxy, cloud LB controllers; the thing that actually does the work
- **The Ingress resource anatomy** — hosts, paths, TLS, `IngressClass`, the `pathType` field
- **The Gateway API** — the newer, more expressive replacement for Ingress
- **NetworkPolicy** — Kubernetes' firewall, by labels and namespaces and CIDRs; ingress and egress rules
- **NetworkPolicy semantics** — the additive-allow model, default-deny patterns, the CNI dependency
- **CoreDNS revisit** — customising the cluster's DNS server

By the end of this notebook you should be able to expose an HTTP service publicly through an Ingress, write a NetworkPolicy that locks down a namespace to only the traffic it needs, and explain why your NetworkPolicy didn't do anything (probably: your CNI doesn't enforce them). Notebooks 04 and 08 are the prerequisites — Services give you the L4 primitive; CNI and `kube-proxy` are what this notebook is asking the CNI to actually implement.

## The Kubernetes networking model — four hard requirements

The Kubernetes networking spec is a small set of axioms. **Every cluster** — your laptop's `kind`, a managed EKS cluster, a 2000-node bare-metal install — must satisfy all four:

1. **Every pod gets its own routable IP.** Not a NAT'd port mapping, not a shared address — a real IP that other pods on any node can dial directly. The pod sees that IP from inside (`hostname -i`, `ip addr`).

2. **Pods can talk to all other pods without NAT.** Whether the other pod is on the same node or a different node, the source IP is preserved end-to-end. From pod A's perspective, talking to pod B is identical regardless of where B is running.

3. **Nodes can talk to all pods (without NAT).** This is what lets the kubelet's probes work, and what lets `kubectl exec` and `kubectl port-forward` proxy through the API server to a pod.

4. **The IP a pod sees for itself is the IP others see it as.** No "internal" vs "external" address inside a pod. The simplicity of this is easy to under-appreciate — Docker's default networking violates it, and that's why running plain Docker at scale gets painful.

Together, these are sometimes called the **"flat" or "IP-per-pod" model**. Compared to the Docker-style "containers share the host IP, expose via NAT" world, it's:

- Easier to reason about — every connection is just IP-to-IP.
- Easier for service meshes and observability — sidecar proxies sit on a real network interface, not behind a NAT.
- Harder to *implement* — the cluster operator has to ensure routing works between every node's pod range and every other node's pod range. That's the CNI's job (notebook 08).

### What Kubernetes *doesn't* mandate

- It doesn't say *how* you achieve the four properties. Overlay tunnels, BGP routing on flat L2, cloud VPC routes, native eBPF — all valid.
- It doesn't say anything about *security*. By default, every pod can reach every other pod in the cluster, in any namespace. NetworkPolicy (later in this notebook) is how you change that.
- It doesn't say anything about *bandwidth* or *quality of service*. Some CNIs offer it, none guarantee it.

## Pod IPs and pod CIDRs — where the addresses come from

A typical kubeadm cluster has two address ranges separate from your VPC or LAN:

- **Pod CIDR** — the range pods get IPs from. Default `192.168.0.0/16` if you `kubeadm init --pod-network-cidr=192.168.0.0/16`. Often `10.244.0.0/16` (Flannel default), `10.42.0.0/16` (Calico default).
- **Service CIDR** — the range Service ClusterIPs come from. Default `10.96.0.0/12`. You met this in notebook 04.

The pod CIDR is **subdivided per node**. Each node gets a slice — typically `/24` (256 IPs) — that it allocates from for the pods it runs. `kubectl get nodes -o wide` shows it; `kubectl get node <name> -o jsonpath='{.spec.podCIDR}'` shows the slice directly.

```
              cluster pod CIDR: 192.168.0.0/16
                          |
        +-----------------+-----------------+
        |                                   |
   node-1: 192.168.1.0/24            node-2: 192.168.2.0/24
        |                                   |
   pods 192.168.1.5 .6 .7 ...        pods 192.168.2.4 .5 ...
```

That layout lets the cluster route by *node*: "any packet destined for `192.168.2.0/24` goes to `node-2`." The CNI's job is to publish those per-node routes — via BGP, overlay tunnel headers, or whatever mechanism it picked.

### Two clusters, same CIDR — the everyday problem

`192.168.0.0/16` overlaps with what corporate LANs and home routers use. If your laptop's `kind` cluster picks the same range as your VPN, traffic to `192.168.x.y` from the host won't know whether it means "the cluster" or "the office." That's why production clusters pick non-overlapping ranges, and tools like Calico let you allocate IPs from a non-default range.

## What a CNI plugin actually does — second-by-second

When the kubelet creates a pod (notebook 08), here's what happens on the networking side:

1. **The kubelet creates the pod's network namespace.** The container runtime does the system-call work; the kubelet asks for it via CRI. The pod now has its own kernel net namespace with no interfaces in it except a loopback.

2. **The kubelet invokes the configured CNI plugin** by running its binary. Inputs (via stdin JSON and env vars): the pod's namespace, the container ID, the CNI config from `/etc/cni/net.d/`.

3. **The CNI plugin allocates an IP** from the node's slice of the pod CIDR (managed by an IPAM plugin — usually `host-local` or `calico-ipam`).

4. **The CNI plugin creates a virtual interface pair** — a `veth` pair. One end goes into the pod's network namespace, named `eth0`. The other stays on the host.

5. **The CNI plugin programs routes** — on the host side, on the pod side, and potentially in the cluster's overlay or BGP fabric. The pod can now `ping 1.1.1.1` (subject to NAT for egress).

6. **The CNI plugin returns the assigned IP** to the kubelet. The kubelet puts this in the pod's `status.podIP`.

7. **`kube-proxy` notices the pod is `Ready`** (later, after probes pass) and updates Service endpoints. Cluster DNS starts returning this pod's IP for any selector that matches.

8. **The pod's containers start.** They see `eth0` already configured with their pod IP.

Steps 1 through 6 take milliseconds. Whatever your CNI does — VXLAN encapsulation, BGP route advertisement, cloud VPC route addition, eBPF socket attachment — it has to do it fast.

### Inspecting it

`kubectl get pod -o wide` shows the IP. `kubectl exec <pod> -- ip addr` from inside the pod shows the network namespace's view. On the node itself, `ip link` shows the host end of every `veth` pair (named `vethXXXX`, or `caliXXXX` depending on the CNI).

## The three communication paths

Once the cluster is networked, every connection a pod makes falls into one of three categories. Each is routed differently in the kernel.

### 1. Pod-to-pod (same or different node)

The simplest case. Pod A wants to reach pod B by IP. The packet leaves pod A's `eth0`, goes through its `veth` pair onto the host, and:

- **Same node** — the host bridges directly to the other `veth` and pod B sees it. No CNI overlay, no kube-proxy.
- **Different node** — the host routes the packet according to the CNI's per-node routes. With Calico in BGP mode, it's a plain L3 hop. With Flannel VXLAN, the packet is encapsulated in a UDP wrapper, sent to the destination node, and unwrapped there.

No NAT in either case — the destination pod sees the source pod's IP unchanged.

### 2. Pod-to-Service

Pod A calls `http://api:80`. The kernel sees a packet destined for the Service's ClusterIP (resolved via DNS — notebook 04). `kube-proxy`'s iptables, IPVS, or nftables rules DNAT it to one of the backing pod IPs — picked at random for ClusterIP-type Services.

The destination pod sees the request from the **client pod's IP**, not from the Service IP. The Service IP is virtual; only the rule does any rewriting.

### 3. Pod-to-external (the wider internet)

Pod A wants to fetch `https://example.com`. The packet leaves the pod, hits the host. The pod's source IP isn't routable on the public internet, so:

- **NAT** — the host **source-NATs** the packet to its own IP before sending. Return traffic gets un-NATed on the way back.
- That's why external services see "the node's IP," not "the pod's IP," when a pod calls out. If you want the *pod* IP to be visible (rare), you need extra work — `externalTrafficPolicy: Local` on a Service, an egress gateway in the CNI, or a service mesh.

### A picture

```
   pod-to-pod                pod-to-service                pod-to-external
+-----+    +-----+      +-----+    +---------+         +-----+    +------+
| A   | -> | B   |      | A   | -> | Service | -> pod  | A   | -> | Node | -> internet
+-----+    +-----+      +-----+    | (virtual|         +-----+    | NAT  |
 direct     IP-to-IP    through    | IP)     |          source-NAT to node IP
                        kube-proxy +---------+
                        DNAT to chosen pod
```

These three patterns explain almost every networking question on the CKA: "why is service X unreachable?" turns into "is this path 1, 2, or 3, and which layer is breaking?

In [ ]:
# The pod CIDR for each node
!kubectl get nodes -o custom-columns=NAME:.metadata.name,INTERNAL-IP:.status.addresses[?\(@.type==\"InternalIP\"\)].address,POD-CIDR:.spec.podCIDR
!echo '---'
# A handful of pods and their IPs across the cluster
!kubectl get pods -A -o wide | head -10
!echo '---'
# Inside a pod, the network namespace's view of itself
!kubectl run nettest -i --rm --restart=Never --image=busybox:1.36 --quiet -- \
  sh -c 'echo "--- ip addr (inside pod):"; ip addr | sed -n "1,20p"; echo; echo "--- resolv.conf:"; cat /etc/resolv.conf'

## Why Ingress — Services don't go far enough

You've got Services. They handle L4 routing inside the cluster and, with `type: LoadBalancer`, can expose a workload publicly. So why isn't Service the whole answer?

**One LoadBalancer per Service is expensive and rigid.** In a cloud cluster, each `LoadBalancer` Service provisions a separate cloud LB — an AWS ELB, a GCP forwarding rule. That's:

- Tens of dollars per month per LB.
- A separate public IP (or hostname) per LB. Hard to brand.
- No HTTP-level routing. You can't say "send `/api/*` to the api Service, `/static/*` to the static Service."
- No TLS termination at the LB. The pod has to do TLS itself.
- No host-based routing. You can't share one IP between `api.example.com` and `web.example.com`.

What you want is **one public entry point** in front of *many* Services, with HTTP-aware routing. That's what an **Ingress** gives you.

```
                         +-------------+
   api.example.com    -> |             | -> Service "api"   -> pods
   www.example.com/api* -> |  Ingress   | -> Service "api"   -> pods
   www.example.com/*    -> |  controller| -> Service "web"   -> pods
   shop.example.com     -> | (one LB IP)| -> Service "shop"  -> pods
                         +-------------+
                         handles TLS,
                         host + path
                         routing
```

One LoadBalancer (or NodePort in cheaper setups) — one public IP — fans out to many in-cluster Services based on HTTP host and path.

### Ingress is the *resource*. Ingress controllers do the work.

`Ingress` is just a Kubernetes object — declarative configuration. It does nothing on its own. You have to install an **ingress controller** in the cluster; the controller watches Ingress resources and configures the actual proxy.

In other words: an Ingress resource is a *spec* the controller obeys. Without a controller, an Ingress is just YAML in `etcd`.

## Ingress controllers — the proxies that do the work

Several controllers exist, all read the same `Ingress` resource format, all implement the routing differently.

| Controller | What's inside | Notes |
|---|---|---|
| **`ingress-nginx`** | nginx + reload-on-change | The most-deployed. CNCF project (not affiliated with Nginx Inc's commercial product). |
| **Traefik** | Traefik proxy | Auto-discovery, nice dashboard, popular for development. |
| **HAProxy Ingress** | HAProxy | Performance-focused. |
| **AWS Load Balancer Controller** | AWS ALB | On EKS — provisions a real ALB per Ingress (or one shared via the `alb.ingress.kubernetes.io/group.name` annotation). |
| **GCE / GKE Ingress** | GCP HTTP(S) LB | Default on GKE. |
| **Istio Gateway, Linkerd, Contour** | Envoy | Service-mesh-flavoured Ingress. |

You pick **one** ingress controller per cluster, install it (typically via Helm), and write Ingress resources that target it.

### `IngressClass` — which controller handles which Ingress

A cluster can have *multiple* ingress controllers (for example, an internal-only nginx and a public ALB). Each registers an `IngressClass` resource. Ingresses pick their controller by setting `spec.ingressClassName`:

```yaml
apiVersion: networking.k8s.io/v1
kind: IngressClass
metadata:
  name: nginx
spec:
  controller: k8s.io/ingress-nginx
---
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: web
spec:
  ingressClassName: nginx     # which controller handles me
  rules:
    - ...
```

There can be one default class, marked with the `ingressclass.kubernetes.io/is-default-class: "true"` annotation. Ingresses without an explicit `ingressClassName` use it.

### What gets exposed

The ingress controller is itself a workload. To be reachable from outside the cluster, it's usually fronted by:

- **`Service` of type `LoadBalancer`** in cloud clusters — one cloud LB per controller, not per Ingress resource. Cheap.
- **`Service` of type `NodePort`** in bare-metal or local clusters. Hit any node IP on the NodePort.
- **`hostNetwork: true`** on the controller pod — listens directly on the node's port 80 and 443. Used when nothing else can share the port.

`kind` doesn't ship an ingress controller by default. To experiment, install `ingress-nginx` with kind's special configuration (port mappings in the kind cluster config).

## Anatomy of an Ingress resource

A minimal but realistic Ingress:

```yaml
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: web
  annotations:
    nginx.ingress.kubernetes.io/rewrite-target: /    # controller-specific
spec:
  ingressClassName: nginx
  tls:
    - hosts:
        - www.example.com
      secretName: web-tls                           # type: kubernetes.io/tls (notebook 05)
  rules:
    - host: www.example.com
      http:
        paths:
          - path: /api
            pathType: Prefix
            backend:
              service:
                name: api
                port:
                  number: 80
          - path: /
            pathType: Prefix
            backend:
              service:
                name: web
                port:
                  number: 80
    - host: api.example.com
      http:
        paths:
          - path: /
            pathType: Prefix
            backend:
              service:
                name: api
                port:
                  number: 80
```

Field by field:

**`spec.tls`** — TLS termination. Each entry binds a list of hostnames to a Secret of type `kubernetes.io/tls` (notebook 05). The controller serves HTTPS for those hosts using the cert in the Secret.

**`spec.rules`** — host-based routing. Each rule matches a `Host:` header; pods behind don't have to know about the original host.

**`spec.rules[].http.paths`** — path-based routing within a host. Each path has:

- **`path`** — the prefix or exact path to match.
- **`pathType`** — `Prefix` (the usual choice), `Exact`, or `ImplementationSpecific`. `Prefix` matches `/api`, `/api/`, `/api/foo`; `Exact` matches only `/api`.
- **`backend.service.name` and `port.number`** — which Service to route to.

**Annotations** — anything controller-specific. `ingress-nginx` annotations start with `nginx.ingress.kubernetes.io/`; ALB uses `alb.ingress.kubernetes.io/`. Common cases: rewrite targets, redirect HTTP to HTTPS, custom timeouts, basic auth.

### TLS — the certificate side

The Secret in `spec.tls[].secretName` must be `type: kubernetes.io/tls` and contain `tls.crt` plus `tls.key`. Three ways to get one:

- **Bring your own** — `kubectl create secret tls`.
- **cert-manager** — an add-on that obtains certs from Let's Encrypt automatically (the production default).
- **The controller's annotations** — some controllers (ALB, GKE Ingress) integrate with cloud-side certificate stores; you reference the cert by ARN or ID instead of mounting a Secret.

### The default backend

Requests that don't match any rule go to a **default backend** — usually a tiny pod that returns 404. The controller has a built-in one; you can override it per Ingress or globally.

In [ ]:
# Even without an ingress controller installed, the API server accepts the manifest —
# the object exists in etcd, just nothing is reading it. On a cluster with ingress-nginx
# (or equivalent), the same manifests would serve HTTP traffic at the configured host.
!cat <<'EOF' | kubectl apply -f -
apiVersion: apps/v1
kind: Deployment
metadata:
  name: hello
spec:
  replicas: 2
  selector:
    matchLabels: { app: hello }
  template:
    metadata:
      labels: { app: hello }
    spec:
      containers:
        - name: app
          image: nginx:alpine
---
apiVersion: v1
kind: Service
metadata:
  name: hello
spec:
  selector:
    app: hello
  ports:
    - port: 80
      targetPort: 80
---
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: hello
spec:
  ingressClassName: nginx
  rules:
    - host: hello.example.com
      http:
        paths:
          - path: /
            pathType: Prefix
            backend:
              service:
                name: hello
                port:
                  number: 80
EOF
!sleep 5
!kubectl get ingress hello
!echo '---'
# Without an ingress controller, the ADDRESS column stays empty — nothing is processing this Ingress.
!kubectl describe ingress hello | sed -n '1,18p'
!echo '---'
!kubectl get ingressclasses 2>/dev/null || echo "(no IngressClasses installed)"


## The Gateway API — Ingress, but more

`Ingress` is showing its age. Half of every controller's behaviour lives in **annotations** — controller-specific magic strings — because the resource itself doesn't have fields for things like rate-limiting, header rewrites, or traffic splitting. The **Gateway API** is the official successor.

It splits the single `Ingress` object into three:

- **`GatewayClass`** — describes a kind of gateway implementation (the analogue of `IngressClass`).
- **`Gateway`** — a *physical* listener: an IP, a port, a TLS config. Owned by infrastructure or platform teams.
- **`HTTPRoute`** (and `GRPCRoute`, `TLSRoute`, `TCPRoute`, ...) — route definitions attached to a Gateway. Owned by application teams.

```yaml
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: public
spec:
  gatewayClassName: nginx
  listeners:
    - name: https
      port: 443
      protocol: HTTPS
      tls:
        certificateRefs:
          - name: web-tls
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata:
  name: hello
spec:
  parentRefs:
    - name: public
  hostnames: ["hello.example.com"]
  rules:
    - matches:
        - path:
            type: PathPrefix
            value: /
      backendRefs:
        - name: hello
          port: 80
```

Why bother:

- **Cleaner role split.** Platform admins own Gateways; app teams own HTTPRoutes. RBAC matches the split.
- **Richer matching.** Headers, query parameters, methods — not just host and path prefix.
- **First-class traffic splitting.** Send 90% to v1 and 10% to v2 — a built-in field, not an annotation.
- **Protocol diversity.** TCP, UDP, gRPC, TLS — same model, different Route kinds.

**Status as of 2026.** Gateway API v1 is GA; most major ingress controllers support it side-by-side with Ingress. The CKA still tests `Ingress`. Learn Ingress first; expect to write Gateways in production within the next year or two.

## NetworkPolicy — Kubernetes' built-in firewall

Out of the box, the cluster is *open*: every pod can reach every other pod in every namespace. That's deliberate — orchestration first, security policy as opt-in. **NetworkPolicy** is the opt-in.

A NetworkPolicy is a namespaced object that says, for the pods matching a selector:

- **Ingress** — which sources can talk *to* them.
- **Egress** — which destinations they can talk *to*.

```yaml
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: web-allow-from-frontend
  namespace: default
spec:
  podSelector:
    matchLabels:
      app: web
  policyTypes:
    - Ingress
    - Egress
  ingress:
    - from:
        - podSelector:
            matchLabels:
              role: frontend
        - namespaceSelector:
            matchLabels:
              kubernetes.io/metadata.name: api-gateway
      ports:
        - protocol: TCP
          port: 80
  egress:
    - to:
        - podSelector:
            matchLabels:
              app: postgres
      ports:
        - protocol: TCP
          port: 5432
    - to:
        - namespaceSelector: {}      # all namespaces
          podSelector:
            matchLabels:
              k8s-app: kube-dns
      ports:
        - protocol: UDP
          port: 53
```

What this policy says:

- **Target.** Apply to pods in `default` with `app: web`.
- **Ingress.** Accept TCP/80 from pods labelled `role: frontend` in the same namespace, OR from any pod in the namespace `api-gateway`.
- **Egress.** Allow outbound TCP/5432 to pods labelled `app: postgres`, and outbound UDP/53 to CoreDNS (in any namespace). Everything else is implicitly denied.

### `from` and `to` are *or* lists; sub-fields are *and*

A common mistake. In the ingress rule above, the `from` list has two entries — they're OR'd. *Within* an entry, the selectors are AND'd:

```yaml
from:
  - namespaceSelector: { matchLabels: { team: a } }
    podSelector:        { matchLabels: { role: frontend } }
```

That matches **pods labelled `role: frontend` that are in namespaces labelled `team: a`**, not "any pod in team-a namespaces OR any frontend pod."

### IP block selectors

You can also match by CIDR — useful for letting in (or out) external traffic:

```yaml
ingress:
  - from:
      - ipBlock:
          cidr: 10.0.0.0/8
          except:
            - 10.0.5.0/24
    ports:
      - protocol: TCP
        port: 443
```

## NetworkPolicy semantics — additive allow, default deny

Three rules to internalise.

### 1. NetworkPolicy is "default-deny once any policy applies"

If **no** NetworkPolicy selects a pod, that pod's traffic is **not restricted** — the default cluster allow-all stands. The moment **one** NetworkPolicy selects the pod (matches `spec.podSelector`), that pod becomes default-deny for the direction(s) the policy lists in `policyTypes`, and **only** the explicitly allowed traffic gets through.

So `policyTypes: [Ingress]` on a policy with empty `ingress: []` is *deny-all-ingress* for the matching pods. That's the "default deny" pattern:

```yaml
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: default-deny-ingress
  namespace: default
spec:
  podSelector: {}      # every pod in this namespace
  policyTypes: [Ingress]
  # no ingress rules — nothing is allowed in
```

### 2. Multiple policies are *additive* — a union

If two policies select the same pod, traffic is allowed if **any** of them allow it. There's no "deny" rule, no precedence, no override — just "is at least one of my policies happy with this packet?" If yes, allow.

That's why you build up coverage with multiple narrowly-scoped policies: a default-deny plus targeted allows for specific apps.

### 3. NetworkPolicy enforcement is the CNI's job

The Kubernetes API server stores NetworkPolicy objects. The actual *enforcement* — the iptables / IPVS / eBPF rules that drop or allow packets — is done by the CNI plugin. **Not every CNI implements NetworkPolicy:**

| CNI | NetworkPolicy support |
|---|---|
| **Calico** | Yes — the reference implementation, extended with cluster-wide rules |
| **Cilium** | Yes — adds L7-aware policies, DNS-aware policies |
| **Flannel** (alone) | **No** — must combine with Calico (`canal`) for policy |
| **Weave Net** | Yes (limited) |
| **kindnet** (`kind`'s default) | **No** |

If your cluster's CNI doesn't support NetworkPolicy, you can `kubectl apply` policies happily — the API server accepts them, they show up in `kubectl get netpol` — and **nothing changes**. Test that your policy actually does something, on a CNI that enforces it.

### A short test you can run after applying a policy

Spin up a sender pod with the right labels and verify connectivity to the target with `wget --timeout=2 ...` or `nc -w 2 ...`. Repeat without the right labels; you should get a timeout.

In [ ]:
# Apply two NetworkPolicies in a fresh namespace.
# On kind's default CNI (kindnet), these are CREATED but NOT ENFORCED.
# On a Calico or Cilium cluster, the same manifests would actually block traffic.
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Namespace
metadata:
  name: np-demo
---
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: default-deny-ingress
  namespace: np-demo
spec:
  podSelector: {}
  policyTypes: [Ingress]
---
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: web-allow-from-frontend
  namespace: np-demo
spec:
  podSelector:
    matchLabels:
      app: web
  policyTypes: [Ingress]
  ingress:
    - from:
        - podSelector:
            matchLabels:
              role: frontend
      ports:
        - protocol: TCP
          port: 80
EOF
!echo '---'
!kubectl get networkpolicies -n np-demo
!echo '---'
!kubectl describe networkpolicy web-allow-from-frontend -n np-demo
!echo '---'
# Reality check: which CNI is on this cluster? kindnet does NOT enforce NetworkPolicy.
!kubectl get pods -n kube-system -o name | grep -iE 'calico|cilium|kindnet|flannel|weave' | head -5 || \
  echo "(no recognisable CNI pods — check kube-system)"


## NetworkPolicy patterns — common recipes

A handful of policies cover 80% of real-world needs. Bookmark these.

### Default deny everything in a namespace

The single most useful policy. Run this on a fresh namespace and *nothing* talks until you write an allow rule for it.

```yaml
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: default-deny-all
  namespace: team-a
spec:
  podSelector: {}
  policyTypes: [Ingress, Egress]
```

### Allow DNS to CoreDNS

Almost everything needs DNS. The exception that's easy to forget — once you've default-denied egress, you have to allow this back.

```yaml
egress:
  - to:
      - namespaceSelector:
          matchLabels:
            kubernetes.io/metadata.name: kube-system
        podSelector:
          matchLabels:
            k8s-app: kube-dns
    ports:
      - protocol: UDP
        port: 53
      - protocol: TCP
        port: 53
```

### Allow only from the same namespace

Lock a namespace's pods so they only accept ingress from siblings:

```yaml
ingress:
  - from:
      - podSelector: {}        # everything in this namespace
```

### Allow from a specific namespace

Use the well-known label `kubernetes.io/metadata.name`, which the API server adds automatically to every namespace:

```yaml
ingress:
  - from:
      - namespaceSelector:
          matchLabels:
            kubernetes.io/metadata.name: frontend
```

### Allow egress only to a specific Service

Tricky, because policies select on *pod* labels, not Service IPs. The standard pattern is to select the *backend pods* by the labels they carry:

```yaml
egress:
  - to:
      - podSelector:
          matchLabels:
            app: payments
    ports:
      - protocol: TCP
        port: 8080
```

This works because the pods backing the `payments` Service have `app: payments`. The policy lets traffic through regardless of whether the client used the Service VIP or hit the pod directly — at the policy layer, both look the same once `kube-proxy` has DNATed.

### Allow egress to the internet, deny internal

```yaml
egress:
  - to:
      - ipBlock:
          cidr: 0.0.0.0/0
          except:
            - 10.0.0.0/8         # not internal
            - 172.16.0.0/12
            - 192.168.0.0/16
```

Useful for sandboxed batch jobs that should never reach your internal services.

### The CKA test pattern

Common exam scenarios: "namespace `team-a` should accept ingress only from namespace `frontend`," or "deployment `payments` should only allow ingress from `frontend` on port 8080 and egress only to the database on 5432." Write the policies; verify with `wget` from a test pod with the right and wrong labels.

## CoreDNS revisit — customising cluster DNS

You met CoreDNS in notebook 04 — every cluster ships it as a Deployment in `kube-system`, fronted by a Service. The default config is fine for 95% of clusters. The cases that warrant customisation are below.

### The CoreDNS ConfigMap

CoreDNS's config — its "Corefile" — lives in a ConfigMap named `coredns` in `kube-system`:

```bash
kubectl edit configmap coredns -n kube-system
```

Typical contents:

```
.:53 {
    errors
    health {
       lameduck 5s
    }
    ready
    kubernetes cluster.local in-addr.arpa ip6.arpa {
       pods insecure
       fallthrough in-addr.arpa ip6.arpa
       ttl 30
    }
    prometheus :9153
    forward . /etc/resolv.conf {
       max_concurrent 1000
    }
    cache 30
    loop
    reload
    loadbalance
}
```

Each line is a CoreDNS *plugin*. The interesting ones:

- **`kubernetes cluster.local`** — the source of all your Service DNS. Reading from the API server.
- **`forward . /etc/resolv.conf`** — anything not in `cluster.local` is forwarded to the node's upstream resolver.
- **`cache 30`** — cache positive responses for 30 seconds.
- **`reload`** — reload the Corefile when the ConfigMap changes (without restarting pods).

### Adding a custom domain — forward to a specific upstream

Common case: pod-to-internal-service over a private hostname (`*.corp.internal`). Add a *server block* in the Corefile:

```
corp.internal:53 {
    forward . 10.0.0.10
    cache 30
}
```

That tells CoreDNS: "for `*.corp.internal`, forward queries to `10.0.0.10` instead of the default upstream." Save the ConfigMap; CoreDNS's `reload` plugin picks it up.

### Per-pod DNS configuration

If you need *one specific pod* to use different DNS settings (different search domains, different nameservers, a custom `ndots`), the pod spec supports `dnsConfig`:

```yaml
spec:
  dnsPolicy: None             # don't inherit from kubelet
  dnsConfig:
    nameservers:
      - 8.8.8.8
    searches:
      - cluster.local
    options:
      - name: ndots
        value: "2"
```

`dnsPolicy` values: `ClusterFirst` (default — use CoreDNS), `Default` (use the node's resolv.conf), `None` (use only `dnsConfig`), `ClusterFirstWithHostNet` (for `hostNetwork` pods).

### `ndots` and the lookup-multiplier problem

The default `ndots: 5` in pod resolv.conf means any name with fewer than 5 dots tries every search domain first. `curl https://google.com` triggers four failing lookups (`google.com.default.svc.cluster.local`, `google.com.svc.cluster.local`, `google.com.cluster.local`, `google.com.<dns-search>`) before the real one. Performance-sensitive workloads override to `ndots: 2`.

## Cleaning up

Delete everything this notebook created so the cluster's tidy for notebook 10:

```bash
kubectl delete deployment hello
kubectl delete service hello
kubectl delete ingress hello
kubectl delete namespace np-demo
```

Or, more broadly:

```bash
kubectl delete deployment,service,ingress --all
kubectl delete namespace np-demo
```